# 01B Retrieve EUR-Lex National Implementing Measures

This workbook reconstructs the NIM / national-implementing-measure layer in explicit stages, including a resumable full-text retrieval step for the national measure documents.

## Editable parameters in this notebook

- NIM search terms shown in the workbook
- whether live NIM notice retrieval is attempted
- whether live NIM full-text retrieval is attempted
- page size / max pages for live web-service calls
- full-text resume / retry controls
- text-length threshold used to classify successful full-text retrieval

In [23]:
from __future__ import annotations

from pathlib import Path
import sys
import importlib

import pandas as pd
from IPython.display import display
import os

ROOT = Path.cwd()
while not (ROOT / 'docs_df_cleaned').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PIPE_DIR = ROOT / 'analysis_pipeline'
PIPE_OUT = PIPE_DIR / 'outputs'
PIPE_OUT.mkdir(parents=True, exist_ok=True)

import analysis_pipeline.functions.eurlex_nim_retrieval as nim_module
import analysis_pipeline.functions.retrieval_queries as retrieval_queries_module
importlib.reload(nim_module)
importlib.reload(retrieval_queries_module)

from analysis_pipeline.functions.retrieval_queries import SEARCH_TERMS_PRIMARY, TRANSLATED_TERMS_PRIMARY
from analysis_pipeline.functions.eurlex_nim_retrieval import (
    select_eligible_celex_acts,
    build_mne_expert_query,
    retrieve_nim_for_acts,
    load_or_fetch_nim,
    summarize_nim_country,
    batch_fetch_nim_fulltext,
    summarize_nim_fulltext_cache_state,
    load_failed_nim_fulltext_attempts,
    load_nim_fulltext_cache_table,
    nim_cache_key_for_row,
    fetch_nim_page_metadata,
)

pd.set_option('display.max_colwidth', 240)
pd.set_option('display.max_columns', 60)
SAVE_OUTPUTS = True


def write_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_OUTPUTS:
        df.to_csv(path, index=index)
    print(f"{'Wrote' if SAVE_OUTPUTS else 'Would write'}: {path}")
    print(f'Shape: {df.shape}')
    display(df.head())


def preview_df(name: str, df: pd.DataFrame, n: int = 5) -> None:
    print(f'{name}: shape={df.shape}')
    display(df.head(n))


EU_DOCS_PATH = PIPE_OUT / 'eurlex_en_fulltext_index.csv'
NIM_CACHE = ROOT / 'data_processed' / 'rq2_eu_national_measures_long.csv'
OUT_ELIGIBLE = PIPE_OUT / '01B_eligible_celex_acts.csv'
OUT_NIM_LONG = PIPE_OUT / '01B_nim_long.csv'
OUT_NIM_COUNTRY = PIPE_OUT / '01B_nim_celex_country.csv'
OUT_NIM_FULLTEXT = PIPE_OUT / '01B_nim_fulltext_long.csv'
NIM_FULLTEXT_CACHE_DIR = PIPE_OUT / 'nim_fulltext_cache'

RUN_LIVE_NIM = True
RUN_LIVE_FULLTEXT = True
PAGE_SIZE = 100
MAX_PAGES = None
LIVE_ACT_LIMIT = None
FULLTEXT_MAX_ROWS = None
FULLTEXT_RESUME = True
FULLTEXT_RETRY_FAILURES = True
FULLTEXT_PROGRESS_EVERY = 5
FULLTEXT_CACHE_EVERY = 50
FULLTEXT_SUCCESS_MIN_CHARS = 500
FULLTEXT_TRACE_ROUTES = False
FULLTEXT_USE_CACHE = False
FULLTEXT_TIMEOUT = (15, 90)
FULLTEXT_RETRIES = 3
FULLTEXT_MIN_INTERVAL_S = 0.5

os.environ["EURLEX_USER"] = "n00esrcw"
os.environ["EURLEX_WEB_PASS"] = "FPfgrb83td0"

## Query terms used in this notebook

In [24]:
SEARCH_TERMS = list(SEARCH_TERMS_PRIMARY)
TRANSLATED_TERMS = {lang: list(terms) for lang, terms in TRANSLATED_TERMS_PRIMARY.items()}

display(pd.DataFrame({'term': SEARCH_TERMS}))
display(
    pd.DataFrame(
        [(lang, term) for lang, terms in TRANSLATED_TERMS.items() for term in terms],
        columns=['language', 'term'],
    ).head(20)
)


,term
0,nature-positive
1,nature positive
2,nature inclusive
3,nature-inclusive
4,nature-inclusive
5,nature inclusive design
6,nature based solutions
7,nature-based solutions
8,biodiversity net gain
9,biodiversity net-gain


,language,term
0,bg,интегриран с природата дизайн
1,bg,природосъобразен дизайн
2,bg,природопозитивен
3,bg,природосъобразни решения
4,bg,стратегия за биологичното разнообразие
5,bg,стратегия на ЕС за биологичното разнообразие
6,bg,възстановяване на природата
7,bg,възстановяване на екосистемите
8,cs,návrh zahrnující přírodu
9,cs,pozitivní pro přírodu


## Select eligible CELEX acts

In [25]:
eu_docs = pd.read_csv(EU_DOCS_PATH, low_memory=False)
eligible = select_eligible_celex_acts(eu_docs)
if LIVE_ACT_LIMIT:
    eligible = eligible.head(LIVE_ACT_LIMIT).copy()
preview_df('eligible', eligible)
display(eligible['eu_act_type'].value_counts(dropna=False).rename_axis('eu_act_type').reset_index(name='acts'))


eligible: shape=(130, 4)


,celex,eu_act_title,eu_act_type,year
0,32004R1590,"Council Regulation (EC) No 1590/2004 of 26 April 2004 establishing a Community programme on the conservation, characterisation, collection and utilisation of genetic resources in agriculture and repealing Regulation (EC) No 1467/94",Regulation,2004.0
1,32005D0173,2005/173/EC: Commission Decision of 12 May 2004 on the State aid implemented by Spain for further restructuring aid to the public Spanish shipyards State aid Case C 40/00 (ex NN 61/00) (notified under document number C(2004) 1620) (Text...,Decision,2005.0
2,32009D0607,2009/607/EC: Commission Decision of 9 July 2009 establishing the ecological criteria for the award of the Community eco-label to hard coverings (Notified under document C(2009) 5613) (Text with EEA relevance),Decision,2009.0
3,32010D0477,2010/477/EU: Besluit van de Commissie van 1 september 2010 tot vaststelling van criteria en methodologische standaarden inzake de goede milieutoestand van mariene wateren (Kennisgeving geschied onder nummer C(2010) 5956) Voor de EER rel...,Decision,2010.0
4,32012B0546,"2012/546/UE, Euratom: Decisión del Parlamento Europeo, de 10 de mayo de 2012 , sobre la aprobación de la gestión en la ejecución del presupuesto general de la Unión Europea para el ejercicio 2010, sección III — Comisión",Budget act,2012.0


,eu_act_type,acts
0,Regulation,44
1,Decision,35
2,Recommendation,29
3,Budget act,12
4,Directive,9
5,Communication,1


## Build NIM query table

In [26]:
query_table = eligible.copy()
query_table['expert_query'] = query_table['celex'].map(build_mne_expert_query)
preview_df('query_table', query_table[['celex', 'eu_act_title', 'expert_query']])


query_table: shape=(130, 3)


,celex,eu_act_title,expert_query
0,32004R1590,"Council Regulation (EC) No 1590/2004 of 26 April 2004 establishing a Community programme on the conservation, characterisation, collection and utilisation of genetic resources in agriculture and repealing Regulation (EC) No 1467/94",DTS_SUBDOM = MNE AND MNE_IMPLEMENTS_DIR = 32004R1590*
1,32005D0173,2005/173/EC: Commission Decision of 12 May 2004 on the State aid implemented by Spain for further restructuring aid to the public Spanish shipyards State aid Case C 40/00 (ex NN 61/00) (notified under document number C(2004) 1620) (Text...,DTS_SUBDOM = MNE AND MNE_IMPLEMENTS_DIR = 32005D0173*
2,32009D0607,2009/607/EC: Commission Decision of 9 July 2009 establishing the ecological criteria for the award of the Community eco-label to hard coverings (Notified under document C(2009) 5613) (Text with EEA relevance),DTS_SUBDOM = MNE AND MNE_IMPLEMENTS_DIR = 32009D0607*
3,32010D0477,2010/477/EU: Besluit van de Commissie van 1 september 2010 tot vaststelling van criteria en methodologische standaarden inzake de goede milieutoestand van mariene wateren (Kennisgeving geschied onder nummer C(2010) 5956) Voor de EER rel...,DTS_SUBDOM = MNE AND MNE_IMPLEMENTS_DIR = 32010D0477*
4,32012B0546,"2012/546/UE, Euratom: Decisión del Parlamento Europeo, de 10 de mayo de 2012 , sobre la aprobación de la gestión en la ejecución del presupuesto general de la Unión Europea para el ejercicio 2010, sección III — Comisión",DTS_SUBDOM = MNE AND MNE_IMPLEMENTS_DIR = 32012B0546*


## Run NIM retrieval or load fallback

In [27]:
#nim_mode = 'cached_fallback'
if RUN_LIVE_NIM:
    try:
        nim_long = retrieve_nim_for_acts(
            eligible,
            page_size=PAGE_SIZE,
            max_pages=MAX_PAGES,
            metadata_cache_dir=NIM_FULLTEXT_CACHE_DIR,
        )
        nim_mode = 'live'
    except Exception as exc:
        print('Live NIM retrieval failed; switching to cached fallback.')
        print(str(exc))
        nim_long = load_or_fetch_nim(eligible, NIM_CACHE, run_live=False, metadata_cache_dir=NIM_FULLTEXT_CACHE_DIR)
else:
    nim_long = load_or_fetch_nim(eligible, NIM_CACHE, run_live=False, metadata_cache_dir=NIM_FULLTEXT_CACHE_DIR)

preview_df('nim_long', nim_long)
print('nim_mode =', nim_mode)


nim_long: shape=(851, 23)


,celex,nim_celex,national_measure_id,nim_date,nim_title,nim_title_lang,available_expr_langs3,member_state_iso3,member_state_name,eurlex_url,eu_act_title,eu_act_type,year,governance_gap_score,ecological_ambition_score,design_specificity_score,cellar_uri,nim_resource_uri,number_of_national_measures,nim_title_notice,available_langs,lang_detected,lang_source
0,32014L0052,72014L0052LTU_282159,282159,2020-03-16,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,lt,lit,LTU,Lithuania,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052LTU_282159,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/0caf2b04-6a91-11ea-b735-01aa75ed71a1,http://publications.europa.eu/resource/nim/282159,1,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,LT,lt,metadata
1,32014L0052,72014L0052PRT_249897,249897,2017-06-02,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",pt,por,PRT,Portugal,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052PRT_249897,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/13320b4e-0265-47cd-85f7-3184004d4d1e,http://publications.europa.eu/resource/nim/249897,1,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",PT,pt,metadata
2,32014L0052,72014L0052FIN_247927,247927,2017-05-15,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,fi,fin,FIN,Finland,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052FIN_247927,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/048ddf08-4989-494a-bb34-9c7035171a61,http://publications.europa.eu/resource/nim/247927,1,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,FI,fi,metadata
3,32014L0052,72014L0052BEL_175069,175069,2008-02-29,"VLAAMSE OVERHEID - 21 DECEMBER 2007. - Decreet tot aanvulling van het decreet van 5 april 1995 houdende algemene bepalingen inzake milieubeleid met een titel XVI « Toezicht, handhaving en veiligheidsmaatregelen».",nl,nld,BEL,Belgium,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052BEL_175069,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,h

nim_mode = live


## Inspect raw NIM records

In [28]:
nim_country = summarize_nim_country(nim_long)
preview_df('nim_country', nim_country)

display(pd.DataFrame([
    {'metric': 'Eligible CELEX acts', 'value': len(eligible)},
    {'metric': 'NIM long rows', 'value': len(nim_long)},
    {'metric': 'Unique NIM CELEX', 'value': int(nim_long['nim_celex'].astype(str).nunique()) if 'nim_celex' in nim_long.columns else 0},
    {'metric': 'Member states', 'value': int(nim_long['member_state_name'].dropna().nunique()) if 'member_state_name' in nim_long.columns else 0},
    {'metric': 'Non-empty nim_title', 'value': int(nim_long['nim_title'].fillna('').astype(str).str.strip().ne('').sum()) if 'nim_title' in nim_long.columns else 0},
    {'metric': 'Non-empty nim_title_lang', 'value': int(nim_long['nim_title_lang'].fillna('').astype(str).str.strip().ne('').sum()) if 'nim_title_lang' in nim_long.columns else 0},
    {'metric': 'Detected language rows', 'value': int(nim_long['lang_detected'].fillna('').astype(str).str.strip().ne('').sum()) if 'lang_detected' in nim_long.columns else 0},
]))

if 'lang_detected' in nim_long.columns:
    preview_df('nim_long_language_counts', nim_long['lang_detected'].value_counts(dropna=False).rename_axis('lang_detected').reset_index(name='rows'))
if 'lang_source' in nim_long.columns:
    preview_df('nim_long_lang_source_counts', nim_long['lang_source'].value_counts(dropna=False).rename_axis('lang_source').reset_index(name='rows'))
preview_df('nim_long_missing_titles', nim_long[nim_long['nim_title'].fillna('').astype(str).str.strip().eq('')][[
    'celex', 'nim_celex', 'national_measure_id', 'member_state_name', 'eurlex_url', 'lang_detected', 'lang_source'
]])
preview_df('nim_long_inferred_languages', nim_long[nim_long['lang_source'].fillna('').astype(str).isin(['country_fallback', 'default_en'])][[
    'celex', 'nim_celex', 'national_measure_id', 'member_state_name', 'eurlex_url', 'nim_title', 'nim_title_lang', 'lang_detected', 'lang_source', 'available_expr_langs3', 'available_langs'
]])


nim_country: shape=(111, 3)


,celex,member_state,number_of_national_measures
0,32014L0052,Austria,5
1,32014L0052,Belgium,42
2,32014L0052,Bulgaria,6
3,32014L0052,Croatia,30
4,32014L0052,Cyprus,2


,metric,value
0,Eligible CELEX acts,130
1,NIM long rows,851
2,Unique NIM CELEX,851
3,Member states,28
4,Non-empty nim_title,851
5,Non-empty nim_title_lang,851
6,Detected language rows,851


nim_long_language_counts: shape=(22, 2)


,lang_detected,rows
0,cs,115
1,fi,87
2,sv,86
3,en,78
4,lt,55


...

,lang_source,rows
0,metadata,851


nim_long_missing_titles: shape=(0, 7)


,celex,nim_celex,national_measure_id,member_state_name,eurlex_url,lang_detected,lang_source


nim_long_inferred_languages: shape=(0, 11)


,celex,nim_celex,national_measure_id,member_state_name,eurlex_url,nim_title,nim_title_lang,lang_detected,lang_source,available_expr_langs3,available_langs


## Load existing NIM full-text cache state and build the next run set

In [29]:
nim_long = nim_long.copy()
nim_long['cache_key'] = nim_long.apply(nim_cache_key_for_row, axis=1)

fulltext_cache_state_df, nim_fulltext_cache_df = summarize_nim_fulltext_cache_state(
    NIM_FULLTEXT_CACHE_DIR,
    nim_long,
    success_min_chars=FULLTEXT_SUCCESS_MIN_CHARS,
)
failed_cache_df = load_failed_nim_fulltext_attempts(
    NIM_FULLTEXT_CACHE_DIR,
    success_min_chars=FULLTEXT_SUCCESS_MIN_CHARS,
)

cache_diag_df = pd.DataFrame([
    {'metric': 'total_input_rows', 'value': len(nim_long)},
    {'metric': 'already_successful', 'value': int(fulltext_cache_state_df['cache_state'].eq('successful').sum()) if not fulltext_cache_state_df.empty else 0},
    {'metric': 'previously_failed', 'value': int(fulltext_cache_state_df['cache_state'].eq('failed').sum()) if not fulltext_cache_state_df.empty else 0},
    {'metric': 'still_pending', 'value': int(fulltext_cache_state_df['cache_state'].eq('pending').sum()) if not fulltext_cache_state_df.empty else len(nim_long)},
    {'metric': 'cache_csv_rows', 'value': len(nim_fulltext_cache_df)},
    {'metric': 'cache_status_200', 'value': int(nim_fulltext_cache_df['retrieval_status'].eq(200).sum()) if not nim_fulltext_cache_df.empty else 0},
    {'metric': 'cache_text_ge_threshold', 'value': int(nim_fulltext_cache_df['text_len'].ge(FULLTEXT_SUCCESS_MIN_CHARS).sum()) if not nim_fulltext_cache_df.empty else 0},
])

display(cache_diag_df)
preview_df('fulltext_cache_state_df', fulltext_cache_state_df)
preview_df('failed_cache_df', failed_cache_df)

next_run_df = nim_long.copy()
if FULLTEXT_RESUME and not fulltext_cache_state_df.empty:
    successful_keys = set(fulltext_cache_state_df.loc[fulltext_cache_state_df['cache_state'].eq('successful'), 'cache_key'].astype(str))
    failed_keys = set(fulltext_cache_state_df.loc[fulltext_cache_state_df['cache_state'].eq('failed'), 'cache_key'].astype(str))
    keep_mask = ~next_run_df['cache_key'].astype(str).isin(successful_keys)
    if not FULLTEXT_RETRY_FAILURES:
        keep_mask &= ~next_run_df['cache_key'].astype(str).isin(failed_keys)
    next_run_df = next_run_df.loc[keep_mask].copy()
if FULLTEXT_MAX_ROWS:
    next_run_df = next_run_df.head(FULLTEXT_MAX_ROWS).copy()

preview_df('next_run_df', next_run_df)


,metric,value
0,total_input_rows,851
1,already_successful,4
2,previously_failed,35
3,still_pending,812
4,cache_csv_rows,51
5,cache_status_200,3
6,cache_text_ge_threshold,3


fulltext_cache_state_df: shape=(851, 15)


,cache_key,nim_celex,national_measure_id,member_state_iso3,member_state_name,lang,retrieval_status,retrieval_error,text_len,source_url,timestamp,file_text_len,text_path,file_exists,cache_state
0,282159,72014L0052LTU_282159,282159,LTU,Lithuania,EN,202,HTTP 202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052LTU_282159%3AEN%3AHTML,2026-03-26T14:18:57.421514+00:00,0,NaN,False,failed
1,249897,72014L0052PRT_249897,249897,PRT,Portugal,EN,202,HTTP 202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052PRT_249897%3AEN%3AHTML,2026-03-26T14:18:57.421514+00:00,0,NaN,False,failed
2,247927,72014L0052FIN_247927,247927,FIN,Finland,EN,202,HTTP 202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052FIN_247927%3AEN%3AHTML,2026-03-26T14:18:57.421514+00:00,0,NaN,False,failed
3,175069,72014L0052BEL_175069,175069,BEL,Belgium,EN,202,HTTP 202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052BEL_175069%3AEN%3AHTML,2026-03-26T14:18:57.421514+00:00,0,NaN,False,failed
4,263166,72014L0052FIN_263166,263166,FIN,Finland,EN,202,HTTP 202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052FIN_263166%3AEN%3AHTML,2026-03-26T14:18:57.421514+00:00,0,NaN,False,failed


failed_cache_df: shape=(48, 27)


,cache_key,celex,nim_celex,national_measure_id,member_state_iso3,member_state_name,nim_title,nim_title_notice,nim_title_lang,eurlex_url,nim_resource_uri,available_expr_langs3,lang,lang_detected,lang_source,available_languages,page_title,page_title_lang,route_used,text_route_used,source_format,text,source_url,retrieval_status,retrieval_error,text_len,timestamp
0,127789,32014L0052,72014L0052SVK_127789,127789,SVK,Slovakia,,,,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052SVK_127789,,,EN,,,,,,,,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052SVK_127789%3AEN%3AHTML,202,HTTP 202,0,2026-03-26T14:18:57.421514+00:00
1,175069,32014L0052,72014L0052BEL_175069,175069,BEL,Belgium,,,,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052BEL_175069,,,EN,,,,,,,,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052BEL_175069%3AEN%3AHTML,202,HTTP 202,0,2026-03-26T14:18:57.421514+00:00
2,202301148,32014L0052,72014L0052BEL_202301148,202301148,BEL,Belgium,,,,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052BEL_202301148,,,EN,,,,,,,,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052BEL_202301148%3AEN%3AHTML,202,HTTP 202,0,2026-03-26T14:18:57.421514+00:00
3,202306673,32022L2523,72022L2523SWE_202306673,202306673,SWE,Sweden,,,,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72022L2523SWE_202306673,,,EN,,,,,,,,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72022L2523SWE_202306673%3AEN%3AHTML,202,HTTP 202,0,2026-03-26T14:11:34.572553+00:00
4,202306673,32022L2523,72022L2523SWE_202306673,202306673,SWE,Sweden,Lag om tilläggsskatt,,sv,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72022L2523SWE_202306673,http://publications.europa.eu/resource/nim/202306673,swe,en,en,retry_sequence,sv,Lag om tilläggsskatt,sv,fallback_generic/lexuriserv,fallback_generic/lexuriserv,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72022L2523SWE_202306673%3AEN%3AHTML,202,HTTP 202,0,2026-04-07T09:33:29.234437+00:00


next_run_df: shape=(847, 24)


,celex,nim_celex,national_measure_id,nim_date,nim_title,nim_title_lang,available_expr_langs3,member_state_iso3,member_state_name,eurlex_url,eu_act_title,eu_act_type,year,governance_gap_score,ecological_ambition_score,design_specificity_score,cellar_uri,nim_resource_uri,number_of_national_measures,nim_title_notice,available_langs,lang_detected,lang_source,cache_key
0,32014L0052,72014L0052LTU_282159,282159,2020-03-16,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,lt,lit,LTU,Lithuania,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052LTU_282159,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/0caf2b04-6a91-11ea-b735-01aa75ed71a1,http://publications.europa.eu/resource/nim/282159,1,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,LT,lt,metadata,282159
1,32014L0052,72014L0052PRT_249897,249897,2017-06-02,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",pt,por,PRT,Portugal,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052PRT_249897,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/13320b4e-0265-47cd-85f7-3184004d4d1e,http://publications.europa.eu/resource/nim/249897,1,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",PT,pt,metadata,249897
2,32014L0052,72014L0052FIN_247927,247927,2017-05-15,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,fi,fin,FIN,Finland,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052FIN_247927,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/048ddf08-4989-494a-bb34-9c7035171a61,http://publications.europa.eu/resource/nim/247927,1,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,FI,fi,metadata,247927
3,32014L0052,72014L0052BEL_175069,175069,2008-02-29,"VLAAMSE OVERHEID - 21 DECEMBER 2007. - Decreet tot aanvulling van het decreet van 5 april 1995 houdende algemene bepalingen inzake milieubeleid met een titel XVI « Toezicht, handhaving en veiligheidsmaatregelen».",nl,nld,BEL,Belgium,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052BEL_175069,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Di

## Debug sample NIM page extraction

In [30]:
nim_page_debug_rows = []
for _, row in next_run_df.head(5).iterrows():
    meta = fetch_nim_page_metadata(row)
    links = meta.get('page_links', {}) or {}
    direct_links = links.get('direct_text_links', [])
    eli_links = links.get('national_website_eli_links', [])
    web_links = links.get('national_website_links', [])
    machine_links = links.get('machine_translation_links', [])
    nim_page_debug_rows.append({
        'nim_celex': row.get('nim_celex', ''),
        'member_state_name': row.get('member_state_name', ''),
        'nim_page_url': meta.get('nim_page_url', ''),
        'nim_rdf_url': meta.get('nim_rdf_url', ''),
        'title_from_page': meta.get('title', ''),
        'title_lang_from_page': meta.get('title_lang', ''),
        'available_languages': ', '.join(meta.get('available_languages', [])),
        'direct_text_links_n': len(direct_links),
        'national_website_eli_links_n': len(eli_links),
        'national_website_links_n': len(web_links),
        'machine_translation_links_n': len(machine_links),
        'direct_text_links_sample': ' | '.join(link.get('url', '') for link in direct_links[:3]),
        'national_website_eli_sample': ' | '.join(link.get('url', '') for link in eli_links[:3]),
        'national_website_sample': ' | '.join(link.get('url', '') for link in web_links[:3]),
        'machine_translation_sample': ' | '.join(link.get('url', '') for link in machine_links[:3]),
    })
nim_page_debug_df = pd.DataFrame(nim_page_debug_rows)
preview_df('nim_page_debug_df', nim_page_debug_df)


nim_page_debug_df: shape=(5, 15)


,nim_celex,member_state_name,nim_page_url,nim_rdf_url,title_from_page,title_lang_from_page,available_languages,direct_text_links_n,national_website_eli_links_n,national_website_links_n,machine_translation_links_n,direct_text_links_sample,national_website_eli_sample,national_website_sample,machine_translation_sample
0,72014L0052LTU_282159,Lithuania,http://publications.europa.eu/resource/nim/282159,http://publications.europa.eu/resource/cellar/0caf2b04-6a91-11ea-b735-01aa75ed71a1/rdf/object/full,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,lt,lt,0,0,0,0,,,,
1,72014L0052PRT_249897,Portugal,http://publications.europa.eu/resource/nim/249897,http://publications.europa.eu/resource/cellar/13320b4e-0265-47cd-85f7-3184004d4d1e/rdf/object/full,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",pt,pt,0,0,0,0,,,,
2,72014L0052FIN_247927,Finland,http://publications.europa.eu/resource/nim/247927,http://publications.europa.eu/resource/cellar/048ddf08-4989-494a-bb34-9c7035171a61/rdf/object/full,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,fi,fi,0,0,0,0,,,,
3,72014L0052BEL_175069,Belgium,http://publications.europa.eu/resource/nim/175069,http://publications.europa.eu/resource/cellar/2021a125-6f35-4ced-8517-a138a6151ed2/rdf/object/full,"VLAAMSE OVERHEID - 21 DECEMBER 2007. - Decreet tot aanvulling van het decreet van 5 april 1995 houdende algemene bepalingen inzake milieubeleid met een titel XVI « Toezicht, handhaving en veiligheidsmaatregelen».",nl,nl,0,0,0,0,,,,
4,72014L0052FIN_263166,Finland,http://publications.europa.eu/resource/nim/263166,http://publications.europa.eu/resource/cellar/15403d2e-d4ec-4cd4-aaa9-9404b4c2341f/rdf/object/full,Landskapslagen om allmänna handlingars offentlighet (1977:72) 30/03/1977,sv,sv,0,0,0,0,,,,


## Batch NIM full-text retrieval

In [31]:
nim_fulltext_mode = 'skipped'
nim_fulltext_live_df = pd.DataFrame()
if RUN_LIVE_FULLTEXT:
    try:
        nim_fulltext_live_df = batch_fetch_nim_fulltext(
            next_run_df,
            cache_dir=NIM_FULLTEXT_CACHE_DIR,
            use_cache=FULLTEXT_USE_CACHE,
            timeout=FULLTEXT_TIMEOUT,
            retries=FULLTEXT_RETRIES,
            min_interval_s=FULLTEXT_MIN_INTERVAL_S,
            verbose=True,
            trace_routes=FULLTEXT_TRACE_ROUTES,
            resume=FULLTEXT_RESUME,
            retry_failures=FULLTEXT_RETRY_FAILURES,
            progress_every=FULLTEXT_PROGRESS_EVERY,
            cache_every=FULLTEXT_CACHE_EVERY,
            success_min_chars=FULLTEXT_SUCCESS_MIN_CHARS,
        )
        nim_fulltext_mode = 'live'
    except Exception as exc:
        print('Live NIM full-text retrieval failed.')
        print(str(exc))
        nim_fulltext_mode = 'live_failed'
else:
    print('RUN_LIVE_FULLTEXT = False; skipping live full-text retrieval.')

print('nim_fulltext_mode =', nim_fulltext_mode)
preview_df('nim_fulltext_live_df', nim_fulltext_live_df)


=== NIM FULLTEXT RESUME STATE ===
Total input rows: 847
Already successful: 0
Previously failed: 35
Still pending: 812
Run set size: 847
Retry failures: True
[NIM TEXT] 1/847 NIM=72014L0052LTU_282159 state=LTU route=fallback_generic/lexuriserv format= status=202 title="Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. ?sakymas Nr. D1-142 ?D" FAIL reason=HTTP 202 length=0
[NIM TEXT] 2/847 NIM=72014L0052PRT_249897 state=PRT route=fallback_generic/lexuriserv format= status=202 title="Lei n.? 37/2017, de 2 de junho, que torna obrigat?ria a avalia??o de impacte amb" FAIL reason=HTTP 202 length=0
[NIM TEXT] 3/847 NIM=72014L0052FIN_247927 state=FIN route=fallback_generic/lexuriserv format= status=202 title="Valtioneuvoston asetus ymp?rist?vaikutusten arviointimenettelyst? / Statsr?dets " FAIL reason=HTTP 202 length=0
[NIM TEXT] 4/847 NIM=72014L0052BEL_175069 state=BEL route=fallback_generic/lexuriserv format= status=202 title="VLAAMSE OVERHEID - 21 DECEMBER 2007. - Decreet tot aanvull

,celex,nim_celex,national_measure_id,member_state_iso3,member_state_name,nim_title,nim_title_notice,nim_title_lang,eurlex_url,available_expr_langs3,available_langs,nim_resource_uri,text_source_url,source_url,full_text_raw,full_text_clean,text_len,retrieval_status,retrieval_error,fetch_seconds,fetched_from_cache,lang,lang_detected,lang_source,text_path,html_path,route_used,text_route_used,content_type,source_format,available_languages,page_title,page_title_lang,cache_key
0,32014L0052,72014L0052LTU_282159,282159,LTU,Lithuania,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,lt,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052LTU_282159,lit,LT,http://publications.europa.eu/resource/nim/282159,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052LTU_282159%3AEN%3AHTML,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052LTU_282159%3AEN%3AHTML,,,0,202,HTTP 202,17.14,False,en,en,retry_sequence,,,fallback_generic/lexuriserv,fallback_generic/lexuriserv,text/html; charset=UTF-8,,lt,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,lt,282159
1,32014L0052,72014L0052PRT_249897,249897,PRT,Portugal,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...","Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",pt,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052PRT_249897,por,PT,http://publications.europa.eu/resource/nim/249897,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052PRT_249897%3AEN%3AHTML,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052PRT_249897%3AEN%3AHTML,,,0,202,HTTP 202,5.56,False,en,en,retry_sequence,,,fallback_generic/lexuriserv,fallback_generic/lexuriserv,text/html; charset=UTF-8,,pt,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",pt,249897
2,32014L0052,72014L0052FIN_247927,247927,FIN,Finland,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,fi,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052FIN_247927,fin,FI,http://publications.europa.eu/resource/nim/247927,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052FIN_247927%3AEN%3AHTML,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052FIN_247927%3AEN%3AHTML,,,0,202,HTTP 202,7.90,False,en,en,retry_sequence,,,fallback_generic/lexuriserv,fallback_generic/lexuriserv,text/html; charset=UTF-8,,fi,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,fi,247927
3,32014L0052,7

## Build the long NIM full-text dataframe

In [40]:
nim_fulltext_cache_df = load_nim_fulltext_cache_table(NIM_FULLTEXT_CACHE_DIR)
base_cols = [
    'cache_key', 'celex', 'nim_celex', 'national_measure_id', 'member_state_iso3', 'member_state_name',
    'nim_title', 'nim_title_notice', 'nim_title_lang', 'eurlex_url', 'available_expr_langs3',
    'nim_resource_uri', 'available_langs', 'lang_detected', 'lang_source'
]
nim_fulltext_index = nim_long[base_cols].drop_duplicates(subset=['cache_key']).copy()

merge_keys = [
    'cache_key', 'celex', 'nim_celex', 'national_measure_id', 'member_state_iso3', 'member_state_name', 'eurlex_url'
]
nim_fulltext_long = nim_fulltext_index.merge(
    nim_fulltext_cache_df.rename(columns={
        'text': 'full_text_clean',
        'source_url': 'text_source_url',
        'nim_title': 'nim_title_extracted',
    })
)


nim_fulltext_long['nim_title_notice'] = nim_fulltext_long.get('nim_title_notice', nim_fulltext_long['nim_title']).fillna('').astype(str)
nim_fulltext_long['nim_title_extracted'] = nim_fulltext_long.get('nim_title_extracted', '').fillna('').astype(str)
nim_fulltext_long['nim_title'] = nim_fulltext_long['nim_title_extracted'].where(
    nim_fulltext_long['nim_title_extracted'].str.strip().ne(''),
    nim_fulltext_long['nim_title_notice'],
)

for col in ['full_text_clean', 'text_source_url', 'retrieval_error', 'lang', 'lang_detected', 'lang_source', 'timestamp', 'route_used', 'text_route_used', 'source_format', 'page_title', 'page_title_lang', 'available_languages', 'available_langs', 'available_expr_langs3']:
    if col in nim_fulltext_long.columns:
        nim_fulltext_long[col] = nim_fulltext_long[col].fillna('').astype(str)
for col in ['retrieval_status', 'text_len']:
    if col in nim_fulltext_long.columns:
        nim_fulltext_long[col] = pd.to_numeric(nim_fulltext_long[col], errors='coerce').fillna(0).astype(int)

preview_df('nim_fulltext_long', nim_fulltext_long)


nim_fulltext_long: shape=(143, 29)


,cache_key,celex,nim_celex,national_measure_id,member_state_iso3,member_state_name,nim_title,nim_title_notice,nim_title_lang,eurlex_url,available_expr_langs3,nim_resource_uri,available_langs,lang_detected,lang_source,nim_title_extracted,lang,available_languages,page_title,page_title_lang,route_used,text_route_used,source_format,full_text_clean,text_source_url,retrieval_status,retrieval_error,text_len,timestamp
0,247568,32014L0052,72014L0052GBR_247568,247568,GBR,United Kingdom,The Forestry (Environmental Impact Assessment) (Scotland) Regulations 2017,The Forestry (Environmental Impact Assessment) (Scotland) Regulations 2017,en,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052GBR_247568,eng,http://publications.europa.eu/resource/nim/247568,EN,en,metadata,The Forestry (Environmental Impact Assessment) (Scotland) Regulations 2017,en,en,The Forestry (Environmental Impact Assessment) (Scotland) Regulations 2017,en,fallback_generic/lexuriserv,fallback_generic/lexuriserv,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052GBR_247568%3AEN%3AHTML,202,HTTP 202,0,2026-04-07T11:10:59.683227+00:00
1,250839,32014L0052,72014L0052EST_250839,250839,EST,Estonia,Related Resources for XML Schema,Haldusmenetluse seadus,et,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052EST_250839,est,http://publications.europa.eu/resource/nim/250839,ET,et,metadata,Related Resources for XML Schema,et,et,Haldusmenetluse seadus,et,national_website,national_website,html,"XML Schema XML Schema 15 October 2014 Table of contents Introduction Resources Introduction This document describes the XML Schema namespace. It also contains a directory of links to these related resources, using Resource Directory Des...",https://www.w3.org/2001/XMLSchema#anyURI,200,,1203,2026-04-07T11:10:59.683227+00:00
2,250667,32014L0052,72014L0052GBR_250667,250667,GBR,United Kingdom,The Planning (Environmental Impact Assessment) Regulations (Northern Ireland) 2017 (2017 No 83) - Correction,The Planning (Environmental Impact Assessment) Regulations (Northern Ireland) 2017 (2017 No 83) - Correction,en,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052GBR_250667,eng,http://publications.europa.eu/resource/nim/250667,EN,en,metadata,The Planning (Environmental Impact Assessment) Regulations (Northern Ireland) 2017 (2017 No 83) - Correction,en,en,The Planning (Environmental Impact Assessment) Regulations (Northern Ireland) 2017 (2017 No 83) - Correction,en,fallback_generic/lexuriserv,fallback_generic/lexuriserv,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052GBR_250667%3AEN%3AHTML,202,HTTP 202,0,2026-04-07T11:10:59.683227+00:00
3,253521,32014L0052,72014L0052SVN_253521,253521,SVN,Slovenia,Related Resources for XML Schema,Gradbeni zakon,sl,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052SVN_253521,slv,http://publications.europa.eu/resource/nim/253521,SL,sl,metadata,Related Resources for XML Schema,sl,sl,Gradbeni zakon,sl,national_website,national_website,html,"XML Schema XML Schema 15 October 2014 Table of contents Introduction Resources Introduction This document describes the XML Schema namespace. It also contains a directory of links to these related resources, using Resource Directory Des...",https://www.w3.org/2001/XMLSchema#anyURI,200,,1203,2026-04-07T11:10:59.683227+00:00
4,202106526,32014L0052,72014L0052AUT_202106526,202106526,AUT,Austria,RIS - LGBLA_NI_20210817_53 - Landesgesetzblatt authentisch für Niederösterreich,"Änderung des Flurverfassungs-Landesgesetzes 1975 (FLG), LGBl. Nr. 53/2021",de,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052AUT_202106526,deu,http://publications.europa.eu/resource/nim/202106526,DE,de,metadata,RIS - LGBLA_NI_20210817_53 - Landesgesetzblatt authentisch für Niederösterreich,de,de,"Änderung des Flurverfassungs-Landesgesetzes 1975 (FLG), LGBl. Nr. 53/2021",de,national_website_eli,national_website_eli,html,RIS - LGBLA_NI_2021

## Inspect text coverage, language detection, and failed rows

In [33]:
nim_fulltext_diag_df = pd.DataFrame([
    {'metric': 'total_nim_records_attempted', 'value': len(next_run_df)},
    {'metric': 'successful_full_texts', 'value': int(nim_fulltext_long['text_len'].ge(FULLTEXT_SUCCESS_MIN_CHARS).sum()) if not nim_fulltext_long.empty else 0},
    {'metric': 'failed_retrievals', 'value': int(nim_fulltext_long['text_len'].lt(FULLTEXT_SUCCESS_MIN_CHARS).sum()) if not nim_fulltext_long.empty else 0},
    {'metric': 'empty_text', 'value': int(nim_fulltext_long['text_len'].eq(0).sum()) if not nim_fulltext_long.empty else 0},
    {'metric': 'titles_populated_from_html', 'value': int((nim_fulltext_long['nim_title_notice'].fillna('').astype(str).str.strip().eq('') & nim_fulltext_long['nim_title'].fillna('').astype(str).str.strip().ne('')).sum()) if not nim_fulltext_long.empty else 0},
])
display(nim_fulltext_diag_df)

url_lang_examples_df = nim_fulltext_long[[
    c for c in ['nim_celex', 'member_state_name', 'eurlex_url', 'lang_detected', 'lang_source', 'lang', 'available_languages']
    if c in nim_fulltext_long.columns
]].copy()
preview_df('url_lang_examples_df', url_lang_examples_df)

detected_lang_counts_df = nim_fulltext_long['lang_detected'].fillna('').astype(str).value_counts(dropna=False).rename_axis('lang_detected').reset_index(name='rows')
preview_df('detected_lang_counts_df', detected_lang_counts_df)

lang_source_counts_df = nim_fulltext_long['lang_source'].fillna('').astype(str).value_counts(dropna=False).rename_axis('lang_source').reset_index(name='rows')
preview_df('lang_source_counts_df', lang_source_counts_df)

route_counts_df = nim_fulltext_long['text_route_used'].fillna('').astype(str).value_counts(dropna=False).rename_axis('text_route_used').reset_index(name='rows')
preview_df('route_counts_df', route_counts_df)

successful_route_counts_df = nim_fulltext_long.loc[
    nim_fulltext_long['text_len'].ge(FULLTEXT_SUCCESS_MIN_CHARS), 'text_route_used'
].fillna('').astype(str).value_counts(dropna=False).rename_axis('text_route_used').reset_index(name='successful_rows')
preview_df('successful_route_counts_df', successful_route_counts_df)

newly_titled_df = nim_fulltext_long[
    nim_fulltext_long['nim_title_notice'].fillna('').astype(str).str.strip().eq('') & nim_fulltext_long['nim_title'].fillna('').astype(str).str.strip().ne('')
][[
    c for c in ['nim_celex', 'member_state_name', 'eurlex_url', 'nim_title', 'lang_detected', 'lang_source', 'text_route_used']
    if c in nim_fulltext_long.columns
]].copy()
preview_df('newly_titled_df', newly_titled_df)

still_problematic_df = nim_fulltext_long[
    nim_fulltext_long['nim_title'].fillna('').astype(str).str.strip().eq('') | nim_fulltext_long['lang_source'].fillna('').astype(str).eq('default_en')
][[
    c for c in ['nim_celex', 'member_state_name', 'eurlex_url', 'nim_title_notice', 'nim_title', 'lang_detected', 'lang_source', 'text_route_used', 'text_source_url', 'retrieval_status', 'retrieval_error', 'text_len']
    if c in nim_fulltext_long.columns
]].copy()
preview_df('still_problematic_df', still_problematic_df)

failed_nim_fulltext_df = nim_fulltext_long[nim_fulltext_long['text_len'].lt(FULLTEXT_SUCCESS_MIN_CHARS)].copy()
preview_df('failed_nim_fulltext_df', failed_nim_fulltext_df[[
    c for c in ['celex', 'nim_celex', 'national_measure_id', 'member_state_name', 'retrieval_status', 'retrieval_error', 'route_used', 'text_route_used', 'text_source_url', 'source_format', 'text_len']
    if c in failed_nim_fulltext_df.columns
]])


,metric,value
0,total_nim_records_attempted,847
1,successful_full_texts,101
2,failed_retrievals,784
3,empty_text,784
4,titles_populated_from_html,0


url_lang_examples_df: shape=(885, 7)


,nim_celex,member_state_name,eurlex_url,lang_detected,lang_source,lang,available_languages
0,72014L0052LTU_282159,Lithuania,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052LTU_282159,lt,metadata,EN,
1,72014L0052LTU_282159,Lithuania,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052LTU_282159,lt,metadata,en,lt
2,72014L0052PRT_249897,Portugal,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052PRT_249897,pt,metadata,EN,
3,72014L0052PRT_249897,Portugal,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052PRT_249897,pt,metadata,en,pt
4,72014L0052FIN_247927,Finland,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052FIN_247927,fi,metadata,EN,


detected_lang_counts_df: shape=(22, 2)


,lang_detected,rows
0,cs,113
1,sv,91
2,fi,89
3,en,83
4,lt,58


...

,nim_celex,member_state_name,eurlex_url,nim_title,lang_detected,lang_source,text_route_used


still_problematic_df: shape=(0, 12)


,nim_celex,member_state_name,eurlex_url,nim_title_notice,nim_title,lang_detected,lang_source,text_route_used,text_source_url,retrieval_status,retrieval_error,text_len


failed_nim_fulltext_df: shape=(784, 11)


,celex,nim_celex,national_measure_id,member_state_name,retrieval_status,retrieval_error,route_used,text_route_used,text_source_url,source_format,text_len
0,32014L0052,72014L0052LTU_282159,282159,Lithuania,202,HTTP 202,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052LTU_282159%3AEN%3AHTML,,0
1,32014L0052,72014L0052LTU_282159,282159,Lithuania,202,HTTP 202,fallback_generic/lexuriserv,fallback_generic/lexuriserv,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052LTU_282159%3AEN%3AHTML,,0
2,32014L0052,72014L0052PRT_249897,249897,Portugal,202,HTTP 202,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052PRT_249897%3AEN%3AHTML,,0
3,32014L0052,72014L0052PRT_249897,249897,Portugal,202,HTTP 202,fallback_generic/lexuriserv,fallback_generic/lexuriserv,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052PRT_249897%3AEN%3AHTML,,0
4,32014L0052,72014L0052FIN_247927,247927,Finland,202,HTTP 202,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052FIN_247927%3AEN%3AHTML,,0


## Save NIM outputs

In [42]:
write_csv(eligible, OUT_ELIGIBLE, index=False)
write_csv(nim_long, OUT_NIM_LONG, index=False)
write_csv(nim_country, OUT_NIM_COUNTRY, index=False)
write_csv(nim_fulltext_long, OUT_NIM_FULLTEXT, index=False)


Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01B_eligible_celex_acts.csv
Shape: (130, 4)


,celex,eu_act_title,eu_act_type,year
0,32004R1590,"Council Regulation (EC) No 1590/2004 of 26 April 2004 establishing a Community programme on the conservation, characterisation, collection and utilisation of genetic resources in agriculture and repealing Regulation (EC) No 1467/94",Regulation,2004.0
1,32005D0173,2005/173/EC: Commission Decision of 12 May 2004 on the State aid implemented by Spain for further restructuring aid to the public Spanish shipyards State aid Case C 40/00 (ex NN 61/00) (notified under document number C(2004) 1620) (Text...,Decision,2005.0
2,32009D0607,2009/607/EC: Commission Decision of 9 July 2009 establishing the ecological criteria for the award of the Community eco-label to hard coverings (Notified under document C(2009) 5613) (Text with EEA relevance),Decision,2009.0
3,32010D0477,2010/477/EU: Besluit van de Commissie van 1 september 2010 tot vaststelling van criteria en methodologische standaarden inzake de goede milieutoestand van mariene wateren (Kennisgeving geschied onder nummer C(2010) 5956) Voor de EER rel...,Decision,2010.0
4,32012B0546,"2012/546/UE, Euratom: Decisión del Parlamento Europeo, de 10 de mayo de 2012 , sobre la aprobación de la gestión en la ejecución del presupuesto general de la Unión Europea para el ejercicio 2010, sección III — Comisión",Budget act,2012.0


Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01B_nim_long.csv
Shape: (851, 24)


,celex,nim_celex,national_measure_id,nim_date,nim_title,nim_title_lang,available_expr_langs3,member_state_iso3,member_state_name,eurlex_url,eu_act_title,eu_act_type,year,governance_gap_score,ecological_ambition_score,design_specificity_score,cellar_uri,nim_resource_uri,number_of_national_measures,nim_title_notice,available_langs,lang_detected,lang_source,cache_key
0,32014L0052,72014L0052LTU_282159,282159,2020-03-16,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,lt,lit,LTU,Lithuania,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052LTU_282159,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/0caf2b04-6a91-11ea-b735-01aa75ed71a1,http://publications.europa.eu/resource/nim/282159,1,Lietuvos Respublikos aplinkos ministro 2020 m. kovo 12 d. įsakymas Nr. D1-142 „Dėl Lietuvos Respublikos aplinkos ministro 2005 m. gruodžio 28 d. įsakymo Nr. D1-643 „Dėl Angliavandenilių išteklių naudojimo projekto rengimo taisyklių patv...,LT,lt,metadata,282159
1,32014L0052,72014L0052PRT_249897,249897,2017-06-02,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",pt,por,PRT,Portugal,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052PRT_249897,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/13320b4e-0265-47cd-85f7-3184004d4d1e,http://publications.europa.eu/resource/nim/249897,1,"Lei n.º 37/2017, de 2 de junho, que torna obrigatória a avaliação de impacte ambiental nas operações de prospeção, pesquisa e extração de hidrocarbonetos, procedendo à terceira alteração ao Decreto-Lei n.º 151-B/2013, de 31 de outubro, ...",PT,pt,metadata,249897
2,32014L0052,72014L0052FIN_247927,247927,2017-05-15,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,fi,fin,FIN,Finland,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052FIN_247927,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Directive,2014.0,None,None,None,http://publications.europa.eu/resource/cellar/048ddf08-4989-494a-bb34-9c7035171a61,http://publications.europa.eu/resource/nim/247927,1,Valtioneuvoston asetus ympäristövaikutusten arviointimenettelystä / Statsrådets förordning om förfarandet vid miljökonsekvensbedömning (277/2017) 11/05/2017,FI,fi,metadata,247927
3,32014L0052,72014L0052BEL_175069,175069,2008-02-29,"VLAAMSE OVERHEID - 21 DECEMBER 2007. - Decreet tot aanvulling van het decreet van 5 april 1995 houdende algemene bepalingen inzake milieubeleid met een titel XVI « Toezicht, handhaving en veiligheidsmaatregelen».",nl,nld,BEL,Belgium,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052BEL_175069,"Directiva 2014/52/UE del Parlamento Europeo y del Consejo, de 16 de abril de 2014 , por la que se modifica la Directiva 2011/92/UE, relativa a la evaluación de las repercusiones de determinados proyectos públicos y privados sobre el med...",Di

Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01B_nim_celex_country.csv
Shape: (111, 3)


,celex,member_state,number_of_national_measures
0,32014L0052,Austria,5
1,32014L0052,Belgium,42
2,32014L0052,Bulgaria,6
3,32014L0052,Croatia,30
4,32014L0052,Cyprus,2


Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01B_nim_fulltext_long.csv
Shape: (143, 29)


,cache_key,celex,nim_celex,national_measure_id,member_state_iso3,member_state_name,nim_title,nim_title_notice,nim_title_lang,eurlex_url,available_expr_langs3,nim_resource_uri,available_langs,lang_detected,lang_source,nim_title_extracted,lang,available_languages,page_title,page_title_lang,route_used,text_route_used,source_format,full_text_clean,text_source_url,retrieval_status,retrieval_error,text_len,timestamp
0,247568,32014L0052,72014L0052GBR_247568,247568,GBR,United Kingdom,The Forestry (Environmental Impact Assessment) (Scotland) Regulations 2017,The Forestry (Environmental Impact Assessment) (Scotland) Regulations 2017,en,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052GBR_247568,eng,http://publications.europa.eu/resource/nim/247568,EN,en,metadata,The Forestry (Environmental Impact Assessment) (Scotland) Regulations 2017,en,en,The Forestry (Environmental Impact Assessment) (Scotland) Regulations 2017,en,fallback_generic/lexuriserv,fallback_generic/lexuriserv,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052GBR_247568%3AEN%3AHTML,202,HTTP 202,0,2026-04-07T11:10:59.683227+00:00
1,250839,32014L0052,72014L0052EST_250839,250839,EST,Estonia,Related Resources for XML Schema,Haldusmenetluse seadus,et,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052EST_250839,est,http://publications.europa.eu/resource/nim/250839,ET,et,metadata,Related Resources for XML Schema,et,et,Haldusmenetluse seadus,et,national_website,national_website,html,"XML Schema XML Schema 15 October 2014 Table of contents Introduction Resources Introduction This document describes the XML Schema namespace. It also contains a directory of links to these related resources, using Resource Directory Des...",https://www.w3.org/2001/XMLSchema#anyURI,200,,1203,2026-04-07T11:10:59.683227+00:00
2,250667,32014L0052,72014L0052GBR_250667,250667,GBR,United Kingdom,The Planning (Environmental Impact Assessment) Regulations (Northern Ireland) 2017 (2017 No 83) - Correction,The Planning (Environmental Impact Assessment) Regulations (Northern Ireland) 2017 (2017 No 83) - Correction,en,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052GBR_250667,eng,http://publications.europa.eu/resource/nim/250667,EN,en,metadata,The Planning (Environmental Impact Assessment) Regulations (Northern Ireland) 2017 (2017 No 83) - Correction,en,en,The Planning (Environmental Impact Assessment) Regulations (Northern Ireland) 2017 (2017 No 83) - Correction,en,fallback_generic/lexuriserv,fallback_generic/lexuriserv,,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A72014L0052GBR_250667%3AEN%3AHTML,202,HTTP 202,0,2026-04-07T11:10:59.683227+00:00
3,253521,32014L0052,72014L0052SVN_253521,253521,SVN,Slovenia,Related Resources for XML Schema,Gradbeni zakon,sl,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052SVN_253521,slv,http://publications.europa.eu/resource/nim/253521,SL,sl,metadata,Related Resources for XML Schema,sl,sl,Gradbeni zakon,sl,national_website,national_website,html,"XML Schema XML Schema 15 October 2014 Table of contents Introduction Resources Introduction This document describes the XML Schema namespace. It also contains a directory of links to these related resources, using Resource Directory Des...",https://www.w3.org/2001/XMLSchema#anyURI,200,,1203,2026-04-07T11:10:59.683227+00:00
4,202106526,32014L0052,72014L0052AUT_202106526,202106526,AUT,Austria,RIS - LGBLA_NI_20210817_53 - Landesgesetzblatt authentisch für Niederösterreich,"Änderung des Flurverfassungs-Landesgesetzes 1975 (FLG), LGBl. Nr. 53/2021",de,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:72014L0052AUT_202106526,deu,http://publications.europa.eu/resource/nim/202106526,DE,de,metadata,RIS - LGBLA_NI_20210817_53 - Landesgesetzblatt authentisch für Niederösterreich,de,de,"Änderung des Flurverfassungs-Landesgesetzes 1975 (FLG), LGBl. Nr. 53/2021",de,national_website_eli,national_website_eli,html,RIS - LGBLA_NI_2021